# Ausreißer und Aufnahme-Kontext: helfen sie dem Modell?

**Autor:** jd · **Datum:** 2026-10-20

**Ziel (#197):** Prüfen, ob die Befunde aus `39_jd_datenqualitaet` die Pipeline verbessern:

1. **Markierte Zeilen nur im Trainingsfold entfernen.** Gemeint sind die spektralen Klassen- und NDVI-Ausreißer, berechnet nur auf den Trainingszeilen jedes Folds. Bewertet wird weiter auf allen Zeilen.
2. **AEZ × Monat als Indikatoren** (`use_aez_month`, #114), neu auf 80/20 und mit der kombinierten SVM.
3. **Normalisierung pro Aufnahme** (`use_acquisition_centering`): Vom Spektrum wird das mittlere Trainingsspektrum seiner AEZ × Monat-Zelle abgezogen. Das nimmt den gemeinsamen Versatz einer Aufnahme heraus. Die Markierung „Ausreißer“ selbst kann kein Merkmal sein: Sie braucht das Label, das es bei Testzeilen nicht gibt.

**Kriterium vorab:** Eine Variante wird nur übernommen, wenn sie die CV-BAcc kombiniert um mindestens `MIN_CV_GAIN` (0,005) verbessert und keine seltene Klasse überproportional verliert. Nur dann folgt **eine** Validierung.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from awp2.config import (
    CLASS_OUTLIER_FLAG,
    MIN_CV_GAIN,
    NDVI_OUTLIER_FLAG,
    REDUCTION_BEST_PARAMS_COL,
    REDUCTION_VARIANT_COL,
)
from awp2.data import combined_label, load_split
from awp2.experiment import RunConfig, TuneConfig, out_of_fold_predictions, run
from awp2.models import make_combined_svm
from awp2.preprocessing import PreprocessingConfig
from awp2.quality import quality_flags
from awp2.reduction import CURVE_METRICS, ReductionVariant, fold_scores, reduction_curve

APPROACH = "combined"
base = PreprocessingConfig(use_spectral_standard_scale=True)
grid = {
    "estimator__C": [100.0, 300.0, 1000.0],
    "estimator__gamma": [0.001, 0.003, 0.01, "scale"],
}

## 1. Merkmals-Varianten (je eigenes Tuning)

In [ ]:
variants = [
    ReductionVariant(name="reference", family="all_bands", preprocessing=base),
    ReductionVariant(
        name="aez_month",
        family="all_bands",
        preprocessing=base.model_copy(update={"use_aez_month": True}),
    ),
    ReductionVariant(
        name="acquisition_centering",
        family="all_bands",
        preprocessing=base.model_copy(update={"use_acquisition_centering": True}),
    ),
]
curve = reduction_curve(
    make_combined_svm(),
    variants,
    TuneConfig(
        name="context",
        approach=APPROACH,
        description="Aufnahme-Kontext als Merkmal (#197)",
        param_grid=grid,
        preprocessing=base,
    ),
)
curve[
    [REDUCTION_VARIANT_COL, *CURVE_METRICS, "bacc_combined_std", REDUCTION_BEST_PARAMS_COL]
].round(3)

## 2. Markierte Zeilen nur im Trainingsfold entfernen

Gleiche Einstellungen wie die beste Referenz, damit nur der Filter den Unterschied macht.

In [ ]:
def spectral_outliers(X: pd.DataFrame, y: pd.DataFrame) -> pd.Series:
    flags = quality_flags(X, y)
    return flags[[CLASS_OUTLIER_FLAG, NDVI_OUTLIER_FLAG]].any(axis=1)


reference_params = curve.set_index(REDUCTION_VARIANT_COL).loc[
    "reference", REDUCTION_BEST_PARAMS_COL
]
config = RunConfig(name="outlier_filter", approach=APPROACH, preprocessing=base, track=False)
reference_model = make_combined_svm().set_params(**reference_params)
oof_reference = out_of_fold_predictions(reference_model, config)
oof_filtered = out_of_fold_predictions(reference_model, config, train_filter=spectral_outliers)
filter_scores = pd.DataFrame(
    {
        "Referenz": fold_scores(oof_reference, config).mean(),
        "ohne markierte Zeilen": fold_scores(oof_filtered, config).mean(),
    }
)
filter_scores["Differenz"] = filter_scores["ohne markierte Zeilen"] - filter_scores["Referenz"]
filter_scores.round(4)

Wen trifft der Filter? Anteil entfernter Trainingszeilen pro Klasse (auf dem ganzen Trainingsteil berechnet, nur zur Einordnung).

In [ ]:
split = load_split()
removed = spectral_outliers(split.X_train, split.y_train)
labels = combined_label(split.y_train)
removed.groupby(labels).agg(["mean", "sum", "size"]).sort_values("mean", ascending=False).head(
    8
).round(3)

## 3. Entscheidung nach dem Kriterium

In [ ]:
reference_score = curve.set_index(REDUCTION_VARIANT_COL).loc["reference", "bacc_combined"]
gains = {
    name: score - reference_score
    for name, score in curve.set_index(REDUCTION_VARIANT_COL)["bacc_combined"].items()
    if name != "reference"
}
gains["outlier_filter"] = filter_scores.loc["bacc_combined", "Differenz"]
decision = pd.DataFrame({"Gewinn CV": gains})
decision["übernehmen"] = decision["Gewinn CV"] >= MIN_CV_GAIN
decision.round(4)

Nur wenn eine Merkmals-Variante das Kriterium erfüllt, wird sie **einmal** validiert. Referenz auf demselben Split: `reduction_svm_all_bands` (`dac6bb78`, BAcc kombiniert 0,866).

In [ ]:
winners = [name for name in decision.index[decision["übernehmen"]] if name != "outlier_filter"]
by_name = {variant.name: variant for variant in variants}
rows = curve.set_index(REDUCTION_VARIANT_COL)
validation = {
    name: run(
        make_combined_svm().set_params(**rows.loc[name, REDUCTION_BEST_PARAMS_COL]),
        RunConfig(
            name=f"combined_svm_{name}",
            approach=APPROACH,
            description=f"Aufnahme-Kontext (#197): {name}",
            tuning_run=rows.loc[name, "tuning_run"],
            preprocessing=by_name[name].preprocessing,
        ),
    ).metrics
    for name in winners
}
pd.DataFrame({name: metrics.model_dump() for name, metrics in validation.items()}).T

## Fazit

| Variante | CV kombiniert | Gewinn | Entscheidung |
| --- | --- | --- | --- |
| Referenz | 0,859 ± 0,025 | – | – |
| + AEZ × Monat-Indikatoren | 0,864 ± 0,028 | +0,005 | **übernehmen** (knapp über `MIN_CV_GAIN`) |
| Normalisierung pro Aufnahme | 0,849 ± 0,023 | −0,010 | nicht übernehmen |
| markierte Zeilen im Training entfernen | 0,824 | −0,034 | nicht übernehmen |

**Validierung** (einmal, `17dc9e16`): BAcc Kultur 0,923, Stadium 0,922, **kombiniert 0,869**, Macro-F1 kombiniert 0,859. Gegenüber der Referenz auf demselben Split (`dac6bb78`, 0,866) sind das +0,003, beim Stadium +0,009.

- **Ausreißer entfernen schadet deutlich (−0,034).** Die markierten Zeilen sind echte, abweichende Aufnahmen, und das Modell braucht sie, um ähnliche Testzeilen richtig vorherzusagen. Entscheidung für die Pipeline: **keine Entfernung**.
- **Normalisierung pro Aufnahme schadet (−0,010).** Der Zellmittelwert enthält auch die Kulturen, die in der Zelle wachsen. Wer ihn abzieht, entfernt echtes Signal. Der Versatz einer Aufnahme wird anders besser genutzt: als Merkmal.
- **AEZ × Monat als Indikatoren hilft knapp** und erfüllt das vorab festgelegte Kriterium. Der Gewinn liegt innerhalb der Streuung (± 0,028), CV und Validierung zeigen aber in dieselbe Richtung, vor allem beim Stadium. Für die Challenge passt die Option: Alle Testzellen kommen im Training vor. Bei unbekannten Aufnahmen hilft sie nicht (#170), dann sind alle Indikatoren 0.
- **Für das Endmodell:** kombinierte RBF-SVM mit `use_aez_month=True` (`C=300`, `gamma=0.003`).